In [ ]:
Rc=cross_val_score(lre,x_data[['horsepower']], y_data,cv=2)
Rc.mean()

# cross_val_predict：交叉验证预测函数——注意和 cross_val_score 只差两个字，功能却不同
# cross_val_predict 回答：“模型对每一辆车的预测分别是多少？”（可用于画图、分析哪些车预测得差）
from sklearn.model_selection import cross_val_predict
# yhat 里的预测值不是来自一个统一的模型——第 1 个样本的预测来自第 1 轮的模型，第 100 个样本的预测来自第 3 轮的模型。4 个模型参数略有差异，所以这组预测适合做评估和可视化，不适合直接当成“某个最终模型”的输出去部署
# 新建 4 个同类型模型，分别用 3/4 的数据训练，各自预测自己没见过的那 1/4 数据，最后把所有预测值按原顺序拼成一整个数组
# cross_val_predict(cv=4) 内部大致等价于：
# model1 = LinearRegression()  # 新建，用第2,3,4份数据训练
# model2 = LinearRegression()  # 新建，用第1,3,4份数据训练
# model3 = LinearRegression()  # 新建，用第1,2,4份数据训练
# model4 = LinearRegression()  # 新建，用第1,2,3份数据训练
yhat = cross_val_predict(lre,x_data[['horsepower']], y_data,cv=4)
yhat[0:5]


# 多元线性回归
lr = LinearRegression()
lr.fit(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_train)

yhat_train = lr.predict(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])
yhat_train[0:5]
yhat_test = lr.predict(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])
yhat_test[0:5]

# 直接调用函数，用于画图
# 用于对比两条数据的概率密度分布曲线（典型用法：真实价格 vs 模型预测价格的分布对比）
Title = 'Distribution  Plot of  Predicted Value Using Training Data vs Training Data Distribution'
DistributionPlot(y_train, yhat_train, "Actual Values (Train)", "Predicted Values (Train)", Title)

Title='Distribution  Plot of  Predicted Value Using Test Data vs Data Distribution of Test Data'
DistributionPlot(y_test,yhat_test,"Actual Values (Test)","Predicted Values (Test)",Title)

# 把“直线拟合不了的数据”变成“直线能拟合的数据”。
# 普通线性回归只能拟合直线：
from sklearn.preprocessing import PolynomialFeatures

# 但价格和马力的真实关系往往是曲线。多项式特征的做法是：人为构造新列，比如 degree=3 时把 1 列 horsepower 变成 3 列
# horsepower	x²	x³
# 100	10000	1,000,000
# 150	22500	3,375,000
# 然后依然用普通的线性回归去拟合
# 对x来说这是直线方程，但对原始 horsepower 来说就是一条曲线——这就是“多项式回归”的本质：特征变了，模型没变（所以 notebook 后面还是用 LinearRegression）。
# 变换前：特征只有 1 个
# horsepower	price
# 100	13500
# 150	16500
# 200	18000

# 变换后，变成三个特征  其实就是构造新的列
# horsepower（原特征 x）	x²（新特征）	x³（新特征）	price
# 100	10000	1,000,000	13500
# 150	22500	3,375,000	16500
# 200	40000	8,000,000

# 总结：偷偷把表格加工成三列（x、x²、x³）再给他——他的工作方式一点没变，但他算出的结果已经能画出曲线了
# 回归模型（LinearRegression）从头到尾都是线性的，“弯”是喂给它的特征造出来的。

# 数学上：polyfit ≡ PolynomialFeatures + LinearRegression，殊途同归，都是解决同一个方程问题
# 定位上：polyfit 是 numpy 给的“快速拟合计算器”，适合一行画曲线；sklearn 那套是“可训练、可评估、可调参、可防过拟合”的完整机器学习流水线——本 lab 研究过拟合和 Ridge 正则化，必须用它

# 这个 lab 的主题是模型评估与优化——要反复做 train/test 划分、交叉验证、调 degree、加 Ridge 正则化、网格搜索。
# 	np.polyfit	PolynomialFeatures + LinearRegression
# 风格	一行出结果，返回系数数组	面向对象：fit / predict / score
# 预测接口	p(x_new) 手动调用	poly.predict(x_pr)，统一 API
# 评估指标	没有内置 R²，要手写	score() 直接给 R²
# 多特征支持	❌ 只能处理一维 x	✅ 任意多列，还能生成交互项（horsepower×engine-size）
# 进 sklearn 生态	❌ 没法接	✅ 能接 Pipeline、cross_val_score、GridSearchCV
# 正则化	❌ 没有防过拟合手段	✅ 配 Ridge 后就是 Part 3 的内容


# 场景	                    选谁
# 拿到一组点，想画条趋势线	polyfit（更快更简单）
# 要评估模型“考试”能力、防止过拟合、调参数	sklearn 那套（polyfit 做不了）



# 创建变换器，指定最高次幂为5
pr = PolynomialFeatures(degree=5)
# 1列 → 6列 [1, x, x², x³, x⁴, x⁵]
# fit_transform 的含义
# fit：分析数据（确定要生成哪些列、列名等）
# transform：实际生成新特征矩阵
#注意！！！！为什么这里没有进行升维，没有x1*x2这种项，因为里面的参数只有一个horsepower，没有办法进行x1*x2
x_train_pr = pr.fit_transform(x_train[['horsepower']])
x_test_pr = pr.fit_transform(x_test[['horsepower']])
pr

# 还是那个普通线性回归
# 但喂进去的是多项式特征


#  选择A：喂 x_train（1列）
# poly.fit(x_train, y_train)
#  模型学到：price = b + w·hp          → 一条直线，R² 很低
#
#  选择B：喂 x_train_pr（6列）← notebook 的做法
# poly.fit(x_train_pr, y_train)
#  模型学到：price = b + w₁x + w₂x² + w₃x³ + w₄x⁴ + w₅x⁵  → 一条曲线

#喂给模型的数据用x_train_pr，训练出模型，也就是解出方程
poly = LinearRegression()
poly.fit(x_train_pr, y_train)

#预测也是用变换后的预测值
yhat = poly.predict(x_test_pr)
yhat[0:5]

print("Predicted values:", yhat[0:4])
# 输出：
# 90     9549.0      ← 90 是索引（这辆车在原始表中的行号）
# 135    11199.0
# 143    13860.0
# 160    7775.0
# Name: price, dtype: float64


print("True values:", y_test[0:4].values)
# 输出：
# [ 9549. 11199. 13860.  7775.]    ← 干干净净的数字

#画图
PollyPlot(x_train['horsepower'], x_test['horsepower'], y_train, y_test, poly,pr)
poly.score(x_train_pr, y_train)
poly.score(x_test_pr, y_test)

Rsqu_test = []

order = [1, 2, 3, 4]
for n in order:
    pr = PolynomialFeatures(degree=n)

    x_train_pr = pr.fit_transform(x_train[['horsepower']])

    x_test_pr = pr.fit_transform(x_test[['horsepower']])

    lr.fit(x_train_pr, y_train)

    Rsqu_test.append(lr.score(x_test_pr, y_test))

plt.plot(order, Rsqu_test)
plt.xlabel('order')
plt.ylabel('R^2')
plt.title('R^2 Using Test Data')
# 在坐标 (x=3, y=0.75) 处添加文字 "Maximum R²"——标注最高点出现的位置，提示读者最优 degree 在 3 附近。
plt.text(3, 0.75, 'Maximum R^2 ')

